# Task 3: Text Summarization (CNN/DailyMail)
**CENG 467 — NLU&G Take-Home Midterm | Student ID: 300201071**

Models: LexRank (Extractive), BART (Abstractive)

Metrics: ROUGE-1/2/L, BLEU, METEOR, BERTScore

In [1]:
!pip install -q "datasets<3.0.0" datasets transformers rouge-score bert-score nltk scikit-learn

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 12.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


In [2]:
import random, numpy as np, torch
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
nltk.download('punkt', quiet=True); nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True); nltk.download('omw-1.4', quiet=True)
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score as nltk_meteor
from rouge_score import rouge_scorer
from transformers import BartTokenizer, BartForConditionalGeneration
from tqdm.auto import tqdm

SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

Device: cuda


## 1. Load CNN/DailyMail Subset

In [3]:
ds = load_dataset('cnn_dailymail', '3.0.0')
test = ds['test'].shuffle(seed=SEED).select(range(500))
articles = test['article']
references = test['highlights']
print(f'Test subset: {len(test)} articles')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

Test subset: 500 articles


## 2. LexRank (Extractive)

In [4]:
def lexrank(text, n_sent=3, threshold=0.1):
    sents = sent_tokenize(text)
    if len(sents) <= n_sent: return ' '.join(sents)
    try:
        tfidf = TfidfVectorizer(stop_words='english').fit_transform(sents)
    except ValueError:
        return ' '.join(sents[:n_sent])
    sim = cosine_similarity(tfidf); sim[sim<threshold]=0
    rs = sim.sum(axis=1,keepdims=True); rs[rs==0]=1
    T = sim/rs; n=len(sents); scores=np.ones(n)/n; d=0.85
    for _ in range(100):
        ns = (1-d)/n + d*T.T.dot(scores)
        if np.allclose(scores,ns,atol=1e-6): break
        scores=ns
    top = sorted(np.argsort(scores)[::-1][:n_sent])
    return ' '.join([sents[i] for i in top])

lex_sums = [lexrank(a) for a in tqdm(articles, desc='LexRank')]
print(f'Generated {len(lex_sums)} LexRank summaries')

LexRank:   0%|          | 0/500 [00:00<?, ?it/s]

Generated 500 LexRank summaries


## 3. BART (Abstractive)

In [5]:
bart_tok = BartTokenizer.from_pretrained('facebook/bart-large-cnn')
bart_model = BartForConditionalGeneration.from_pretrained('facebook/bart-large-cnn').to(DEVICE)
bart_model.eval()

bart_sums = []
for art in tqdm(articles, desc='BART'):
    inp = bart_tok(art, return_tensors='pt', max_length=512, truncation=True).to(DEVICE)
    with torch.no_grad():
        ids = bart_model.generate(inp['input_ids'], max_length=150, min_length=30,
                                  num_beams=4, length_penalty=2.0, no_repeat_ngram_size=3)
    bart_sums.append(bart_tok.decode(ids[0], skip_special_tokens=True))
print(f'Generated {len(bart_sums)} BART summaries')

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

BART:   0%|          | 0/500 [00:00<?, ?it/s]

Generated 500 BART summaries


## 4. Evaluation

In [6]:
def eval_sums(preds, refs, name):
    scorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
    r1,r2,rl = [],[],[]
    for p,r in zip(preds,refs):
        s=scorer.score(r,p); r1.append(s['rouge1'].fmeasure); r2.append(s['rouge2'].fmeasure); rl.append(s['rougeL'].fmeasure)
    sm=SmoothingFunction().method1; bleus=[]
    for p,r in zip(preds,refs):
        rt=word_tokenize(r.lower()); pt=word_tokenize(p.lower())
        bleus.append(sentence_bleu([rt],pt,smoothing_function=sm) if pt else 0)
    mets=[]
    for p,r in zip(preds,refs):
        mets.append(nltk_meteor([word_tokenize(r.lower())], word_tokenize(p.lower())))
    from bert_score import score as bscore
    _,_,F1 = bscore(preds, refs, lang='en', verbose=False)
    res = {'rouge1':np.mean(r1),'rouge2':np.mean(r2),'rougeL':np.mean(rl),
           'bleu':np.mean(bleus),'meteor':np.mean(mets),'bertscore':F1.mean().item()}
    print(f'\n{name}:')
    for k,v in res.items(): print(f'  {k}: {v:.4f}')
    return res

lex_res = eval_sums(lex_sums, references, 'LexRank')
bart_res = eval_sums(bart_sums, references, 'BART')

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



LexRank:
  rouge1: 0.3505
  rouge2: 0.1361
  rougeL: 0.2269
  bleu: 0.0816
  meteor: 0.3157
  bertscore: 0.8657


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



BART:
  rouge1: 0.4321
  rouge2: 0.2074
  rougeL: 0.3039
  bleu: 0.1419
  meteor: 0.3699
  bertscore: 0.8811


## 5. Qualitative Examples (3)

In [7]:
for i in range(3):
    print(f'\n{"="*70}')
    print(f'Example {i+1}')
    print(f'{"="*70}')
    print(f'Article (first 300 chars): {articles[i][:300]}...')
    print(f'\nReference: {references[i][:200]}')
    print(f'\nLexRank: {lex_sums[i][:200]}')
    print(f'\nBART: {bart_sums[i][:200]}')


Example 1
Article (first 300 chars): (CNN) I see signs of a revolution everywhere. I see it in the op-ed pages of the newspapers, and on the state ballots in nearly half the country. I see it in politicians who once preferred to play it safe with this explosive issue but are now willing to stake their political futures on it. I see the...

Reference: CNN's Dr. Sanjay Gupta says we should legalize medical marijuana now .
He says he knows how easy it is do nothing "because I did nothing for too long"

LexRank: I see this medical marijuana revolution in surprising places. "Marijuana...?" Your medical marijuana questions answered .

BART: Sally Kohn: I see signs of a revolution everywhere. For the first time a majority, 53%, favor its legalization, says Kohn. Support for legalization has risen 11 points in the past few years alone. Koh

Example 2
Article (first 300 chars): He looks barely teenage. But this child has amassed thousands of Twitter followers with his pictorial updates of 'gan

## 6. Final Results Summary (for LaTeX Report)

In [8]:
print('\n' + '='*70)
print('  TASK 3 — FINAL RESULTS (Copy to LaTeX)')
print('='*70)
print(f'{"Model":<10} {"R-1":<8} {"R-2":<8} {"R-L":<8} {"BLEU":<8} {"METEOR":<8} {"BERTScore":<10}')
print('-'*60)
print(f'{"LexRank":<10} {lex_res["rouge1"]:<8.4f} {lex_res["rouge2"]:<8.4f} {lex_res["rougeL"]:<8.4f} {lex_res["bleu"]:<8.4f} {lex_res["meteor"]:<8.4f} {lex_res["bertscore"]:<10.4f}')
print(f'{"BART":<10} {bart_res["rouge1"]:<8.4f} {bart_res["rouge2"]:<8.4f} {bart_res["rougeL"]:<8.4f} {bart_res["bleu"]:<8.4f} {bart_res["meteor"]:<8.4f} {bart_res["bertscore"]:<10.4f}')
print('-'*60)


  TASK 3 — FINAL RESULTS (Copy to LaTeX)
Model      R-1      R-2      R-L      BLEU     METEOR   BERTScore 
------------------------------------------------------------
LexRank    0.3505   0.1361   0.2269   0.0816   0.3157   0.8657    
BART       0.4321   0.2074   0.3039   0.1419   0.3699   0.8811    
------------------------------------------------------------
